# Go2 Track 2 Bonus Project
**Team: Jiarao_Zhang** | EEC289A/EEC289Q

Run all cells top-to-bottom. Artifacts are auto-saved to Google Drive after each major step so a Colab restart won't lose progress.

**Order of operations:**
1. Mount Drive + Config
2. Install + Clone repos
3. Copy Go2 assets
4. Patch planner.py → VX_MAX = 0.85 m/s
5. Train low-level policy (~2-3 h)
6. CMA-ES MLP training, in-process (~30 min)
7. Full track evaluation
8. Package submission

In [ ]:
# ── CELL 1: Mount Drive + Config ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, subprocess, json, io, tarfile, tempfile, time, urllib.request
from pathlib import Path
from urllib.parse import urlparse

TEAM_NAME            = "Jiarao_Zhang"
COURSE_REPO_URL      = "https://github.com/jiarao76/Final-Project-Track-2-Bonus-Project.git"
COURSE_REPO_BRANCH   = "main"

PLAYGROUND_REPO      = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF       = "dd38c285c6d54266287081e516109f0b15985818"
UNITREE_MUJOCO_REPO  = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF   = "1a37b051a10be723405b7ed6dc839361af036d88"
MENAGERIE_REPO       = "https://github.com/deepmind/mujoco_menagerie.git"
MENAGERIE_REF        = "1b86ece576591213e2b666ebf59508454200ca97"

BASE_DIR             = Path("/content")
COURSE_REPO_DIR      = BASE_DIR / "go2_track_bonus_repo"
PLAYGROUND_DIR       = BASE_DIR / "mujoco_playground"
UNITREE_DIR          = BASE_DIR / "unitree_mujoco"
MENAGERIE_DIR        = PLAYGROUND_DIR / "mujoco_playground" / "external_deps" / "mujoco_menagerie"

DRIVE_BACKUP         = Path("/content/drive/MyDrive/go2_track_backup")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

print("Drive backup dir:", DRIVE_BACKUP)
print("Course repo dir: ", COURSE_REPO_DIR)

def run(cmd):
    cmd = [str(c) for c in cmd]
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)

def github_archive_url(repo_url, ref):
    repo_path = urlparse(repo_url).path.strip("/")
    if repo_path.endswith(".git"):
        repo_path = repo_path[:-4]
    return f"https://codeload.github.com/{repo_path}/tar.gz/{ref}"

def download_repo_snapshot(repo_url, ref, target_dir):
    archive_url = github_archive_url(repo_url, ref)
    print(f"+ download {archive_url}")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f"{target_dir.name}_", dir=str(target_dir.parent)))
    try:
        with urllib.request.urlopen(archive_url) as response:
            payload = response.read()
        with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
            archive.extractall(tmp_dir)
        extracted_dirs = [p for p in tmp_dir.iterdir() if p.is_dir()]
        if len(extracted_dirs) != 1:
            raise RuntimeError(f"Expected one extracted directory, got {extracted_dirs}")
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.move(str(extracted_dirs[0]), str(target_dir))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

def ensure_pinned_repo(repo_url, ref, target_dir):
    if target_dir.exists() and (target_dir / ".git").exists():
        try:
            run(["git", "-C", target_dir, "fetch", "--all", "--tags"])
            run(["git", "-C", target_dir, "checkout", ref])
            return
        except subprocess.CalledProcessError:
            shutil.rmtree(target_dir)
    elif target_dir.exists():
        shutil.rmtree(target_dir)
    try:
        run(["git", "clone", repo_url, target_dir])
        run(["git", "-C", target_dir, "checkout", ref])
    except subprocess.CalledProcessError:
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, ref, target_dir)

def ensure_course_repo(repo_url, branch, target_dir, reset=False):
    if target_dir.exists():
        if reset:
            shutil.rmtree(target_dir)
        else:
            print(f"+ reuse existing course repo at {target_dir}")
            return
    try:
        run(["git", "clone", repo_url, target_dir])
    except subprocess.CalledProcessError:
        if target_dir.exists():
            shutil.rmtree(target_dir)
        download_repo_snapshot(repo_url, branch, target_dir)

print("Config done.")

In [ ]:
# ── CELL 2: Install packages + Clone repos ────────────────────────────────────
if shutil.which("ffmpeg") is None:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "ffmpeg"])

ensure_pinned_repo(PLAYGROUND_REPO,     PLAYGROUND_REF,     PLAYGROUND_DIR)
ensure_pinned_repo(UNITREE_MUJOCO_REPO, UNITREE_MUJOCO_REF, UNITREE_DIR)
ensure_pinned_repo(MENAGERIE_REPO,      MENAGERIE_REF,      MENAGERIE_DIR)
ensure_course_repo(COURSE_REPO_URL, COURSE_REPO_BRANCH, COURSE_REPO_DIR)

os.chdir(COURSE_REPO_DIR)
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground 2>/dev/null || true
!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}

os.chdir(PLAYGROUND_DIR)
!python -m pip install -q -e .
os.chdir(COURSE_REPO_DIR)

if str(PLAYGROUND_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PLAYGROUND_DIR.resolve()))
if str(COURSE_REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(COURSE_REPO_DIR.resolve()))

import jax
import mujoco_playground
print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())

expected_playground = str(PLAYGROUND_DIR.resolve())
if expected_playground not in str(Path(mujoco_playground.__file__).resolve()):
    raise RuntimeError(f"mujoco_playground imported from wrong location")
print("Setup complete.")

In [ ]:
# ── CELL 3: Copy Go2 assets ───────────────────────────────────────────────────
os.chdir(COURSE_REPO_DIR)
!python scripts/copy_go2_assets.py \
    --unitree-dir {UNITREE_DIR} \
    --course-dir {COURSE_REPO_DIR}
print("Assets copied.")

In [ ]:
# ── CELL 4: Patch planner.py – raise speed limits ────────────────────────────
# The repo has VX_MAX=0.50, but the low-level policy is trained to handle
# up to 0.95 m/s (stage_2 config). We raise VX_MAX to 0.85 for faster laps.
planner_py = COURSE_REPO_DIR / "track_bonus" / "planner.py"
content = planner_py.read_text()

replacements = {
    "_VX_MAX: float = 0.50": "_VX_MAX: float = 0.85",
    "_VY_LIM: float = 0.10": "_VY_LIM: float = 0.20",
    "_YAW_LIM: float = 0.30": "_YAW_LIM: float = 0.45",
}
for old, new in replacements.items():
    if old in content:
        content = content.replace(old, new)
        print(f"Patched: {old} → {new}")
    else:
        print(f"[warn] pattern not found (maybe already patched): {old}")
planner_py.write_text(content)

# Verify
for line in content.splitlines():
    if "_VX_MAX" in line or "_VY_LIM" in line or "_YAW_LIM" in line:
        print(" ", line.strip())

In [ ]:
# ── CELL 5: Configure runtime ─────────────────────────────────────────────────
import json
os.chdir(COURSE_REPO_DIR)

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 5_000_000,
}

config_path    = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_cfg_path  = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config    = json.loads(base_cfg_path.read_text())
base_config["runtime_overrides"] = runtime_config
config_path.write_text(json.dumps(base_config, indent=2))
print("Written:", config_path)

# Dry-run to verify
!python train.py --config {config_path} --dry-run

In [ ]:
# ── CELL 6: Train low-level locomotion policy (~2-3 hours) ────────────────────
# Stage 1: forward-only gait  (10M steps)
# Stage 2: full vx/vy/yaw up to 0.95 m/s  (5M steps)
# SKIP this cell if you already have a checkpoint at CHECKPOINT_DIR below.
os.chdir(COURSE_REPO_DIR)
!python train.py \
    --config configs/colab_runtime_config.json \
    --stage both \
    --output-dir artifacts/low_level_train

CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "low_level_train" / "best_checkpoint"
print("Checkpoint exists:", CHECKPOINT_DIR.exists())

In [ ]:
# ── CELL 7: Save checkpoint to Drive (run immediately after Cell 6) ───────────
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "low_level_train" / "best_checkpoint"
drive_ckpt = DRIVE_BACKUP / "best_checkpoint"

if CHECKPOINT_DIR.exists():
    if drive_ckpt.exists():
        shutil.rmtree(drive_ckpt)
    shutil.copytree(str(CHECKPOINT_DIR), str(drive_ckpt))
    print("Checkpoint saved to Drive:", drive_ckpt)
    print("Contents:", [f.name for f in drive_ckpt.iterdir()])
else:
    print("[error] Checkpoint not found. Did Cell 6 finish?")

In [ ]:
# ── CELL 7b: Restore checkpoint from Drive (if Colab restarted) ───────────────
# Run this instead of Cell 6+7 if the session was interrupted.
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "low_level_train" / "best_checkpoint"
drive_ckpt = DRIVE_BACKUP / "best_checkpoint"

if not CHECKPOINT_DIR.exists() and drive_ckpt.exists():
    CHECKPOINT_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(drive_ckpt), str(CHECKPOINT_DIR))
    print("Restored checkpoint from Drive.")
elif CHECKPOINT_DIR.exists():
    print("Checkpoint already present locally.")
else:
    print("[error] No checkpoint found locally or on Drive. Run Cell 6 first.")

print("Checkpoint exists:", CHECKPOINT_DIR.exists())

In [ ]:
# ── CELL 8: Quick policy sanity test (5 s, no render) ────────────────────────
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "low_level_train" / "best_checkpoint"
PLANNER_CONFIG = COURSE_REPO_DIR / "configs" / "starter_planner.json"
SMOKE_DIR      = COURSE_REPO_DIR / "artifacts" / "smoke_test"

os.chdir(COURSE_REPO_DIR)
!python run_track_bonus.py \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --planner-config {PLANNER_CONFIG} \
    --config configs/colab_runtime_config.json \
    --output-dir {SMOKE_DIR} \
    --entry-name {TEAM_NAME} \
    --duration-seconds 10 \
    --no-render

if (SMOKE_DIR / "results.json").exists():
    r = json.loads((SMOKE_DIR / "results.json").read_text())
    print("Smoke test metrics:", r["metrics"])
    print("Policy is working!")
else:
    print("[warn] results.json not found – check errors above")

In [ ]:
# ── CELL 9: In-process CMA-ES MLP training (~20-40 min) ──────────────────────
#
# Loads env + policy ONCE, then evaluates each CMA-ES candidate in-process.
# This avoids the 120s JIT overhead per subprocess call.

import numpy as np, time, json
from pathlib import Path

# Ensure sys.path includes the course repo
if str(COURSE_REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(COURSE_REPO_DIR.resolve()))

from track_bonus.planner import MLPTrackPlanner, _VX_MAX, _VX_MIN, _VY_LIM, _YAW_LIM
from track_bonus.official_track import official_track
from track_bonus.scoring import compute_track_bonus_metrics, score_track_bonus
from course_common import lazy_import_stack, load_json, set_runtime_env
from run_track_bonus import rollout, _make_env
from test_policy import load_policy_with_workaround

print(f"Speed limits: VX=[{_VX_MIN}, {_VX_MAX}]  VY=±{_VY_LIM}  YAW=±{_YAW_LIM}")

CHECKPOINT_DIR  = COURSE_REPO_DIR / "artifacts" / "low_level_train" / "best_checkpoint"
HIGHLEVEL_DIR   = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
HIGHLEVEL_DIR.mkdir(parents=True, exist_ok=True)

# ── CMA-ES hyper-params ────────────────────────────────────────────────────────
HIDDEN_SIZES   = [32, 16]
N_GENERATIONS  = 10
POPULATION     = 8
SIGMA0         = 0.30
SEED           = 42
EVAL_SECONDS   = 350.0   # sim seconds per candidate (enough for 1 lap at 0.7+ m/s)

# ── Load env + policy ONCE ────────────────────────────────────────────────────
print("Loading environment and policy (first rollout triggers JIT)...")
set_runtime_env()
course_cfg = load_json(COURSE_REPO_DIR / "configs" / "colab_runtime_config.json")
course_cfg["runtime_overrides"] = {}
num_steps = int(round(EVAL_SECONDS / course_cfg["control"]["ctrl_dt"]))
track = official_track()
stack = lazy_import_stack()
env   = _make_env(stack, course_cfg, "stage_2", num_steps)
policy = load_policy_with_workaround(CHECKPOINT_DIR.resolve(), deterministic=True)
policy = stack["jax"].jit(policy)
print(f"Env ready. num_steps={num_steps}")

# ── Fitness function ───────────────────────────────────────────────────────────
def evaluate(theta, seed=SEED):
    """Returns (fitness, metrics). Fitness > 1.0 means lap completed."""
    weights = MLPTrackPlanner.unpack(theta, HIDDEN_SIZES)
    planner = MLPTrackPlanner(weights, HIDDEN_SIZES, stand_seconds=1.0)
    result  = rollout(
        stack=stack, env=env, policy=policy, planner=planner,
        track=track, num_steps=num_steps, seed=seed, start_s=0.0, force_cpu=False,
    )
    metrics = compute_track_bonus_metrics(result, track)
    dist    = metrics["valid_distance_m"] / 200.0   # 0..N laps
    fall    = metrics.get("fall", True)
    ft      = metrics.get("finish_time")             # None if incomplete
    fitness = min(dist, 1.0)
    if ft is not None:
        # Completed! Speed bonus: 1.0 at t=0, 0.0 at t=350s
        speed_bonus = max(0.0, (350.0 - float(ft)) / 350.0)
        fitness = 1.0 + speed_bonus
    if fall:
        fitness *= 0.55
    return float(fitness), metrics

# ── Minimal diagonal CMA-ES ───────────────────────────────────────────────────
class _CMAes:
    def __init__(self, x0, sigma0=0.3, popsize=8, seed=0):
        self.rng   = np.random.default_rng(seed)
        self.n     = len(x0)
        self.mean  = x0.copy().astype(np.float64)
        self.sigma = float(sigma0)
        self.lam   = popsize
        self.mu    = max(popsize // 2, 2)
        raw_w      = np.log(self.mu + 0.5) - np.log(np.arange(1, self.mu + 1))
        self.w     = raw_w / raw_w.sum()
        self.mueff = 1.0 / float(np.sum(self.w ** 2))
        self.cs    = (self.mueff + 2.0) / (self.n + self.mueff + 5.0)
        self.ds    = 1.0 + 2.0 * max(0.0, np.sqrt((self.mueff-1.0)/(self.n+1.0))-1.0) + self.cs
        self.chiN  = float(np.sqrt(self.n) * (1.0 - 1.0/(4.0*self.n) + 1.0/(21.0*self.n**2)))
        self.ps    = np.zeros(self.n)
        self.var   = np.ones(self.n)

    def ask(self):
        z = self.rng.standard_normal((self.lam, self.n))
        return self.mean + self.sigma * np.sqrt(self.var) * z

    def tell(self, xs, scores):
        order     = np.argsort(-scores)
        elite     = xs[order[:self.mu]]
        old_mean  = self.mean.copy()
        self.mean = (self.w[:, None] * elite).sum(axis=0)
        step      = (self.mean - old_mean) / (self.sigma * np.sqrt(self.var) + 1e-12)
        self.ps   = (1.0 - self.cs) * self.ps + np.sqrt(self.cs*(2.0-self.cs)*self.mueff) * step
        self.sigma *= float(np.exp((self.cs/self.ds)*(np.linalg.norm(self.ps)/self.chiN - 1.0)))
        self.sigma  = float(np.clip(self.sigma, 1e-8, 2.0))
        ys         = (elite - old_mean) / (self.sigma * np.sqrt(self.var) + 1e-12)
        self.var   = 0.9*self.var + 0.1*float(np.sum(self.w))*(self.w[:,None]*ys**2).sum(axis=0)
        self.var   = np.clip(self.var, 1e-10, None)

# ── Initialise ─────────────────────────────────────────────────────────────────
init_w  = MLPTrackPlanner.make_weights(HIDDEN_SIZES, seed=SEED)
theta0  = MLPTrackPlanner.pack(init_w)
n_params = MLPTrackPlanner.param_count(HIDDEN_SIZES)
print(f"MLP 5→{HIDDEN_SIZES}→3  ({n_params} params)")
print(f"CMA-ES: {N_GENERATIONS} gens × {POPULATION} pop")

es          = _CMAes(theta0, sigma0=SIGMA0, popsize=POPULATION, seed=SEED)
best_score  = -1.0
best_theta  = theta0.copy()
history     = []

print("\nWarm-up (first rollout compiles JAX, may take 2-3 min)...")
t_warmup = time.time()
_, warmup_metrics = evaluate(theta0, seed=SEED)
print(f"Warm-up done in {time.time()-t_warmup:.1f}s")
print("Warm-up metrics:", {k: round(v,3) if isinstance(v,float) else v
                           for k,v in warmup_metrics.items() if k != 'finish_time' or v is not None})

# ── CMA-ES loop ───────────────────────────────────────────────────────────────
for gen in range(N_GENERATIONS):
    t_gen = time.time()
    print(f"\n── Gen {gen+1}/{N_GENERATIONS}  (σ={es.sigma:.4f}) ──")
    candidates = es.ask()
    scores     = np.zeros(POPULATION)

    for idx, theta in enumerate(candidates):
        t_cand = time.time()
        score, m = evaluate(theta, seed=SEED + gen*100 + idx)
        scores[idx] = score
        dist_m  = m["valid_distance_m"]
        fall    = m.get("fall", True)
        ft      = m.get("finish_time")
        star    = "★" if score > best_score else " "
        lap_str = f"LAP {ft:.1f}s" if ft is not None else f"dist={dist_m:.1f}m"
        print(f"  {star} [{idx+1}/{POPULATION}] score={score:.4f}  {lap_str}  fall={fall}  ({time.time()-t_cand:.1f}s)")
        if score > best_score:
            best_score = score
            best_theta = theta.copy()
            best_w = MLPTrackPlanner.unpack(best_theta, HIDDEN_SIZES)
            np.savez(str(HIGHLEVEL_DIR / "planner_weights.npz"), **best_w)

    es.tell(candidates, scores)
    gen_time = time.time() - t_gen
    history.append({
        "generation": gen, "best_score": float(best_score),
        "gen_max": float(scores.max()), "gen_mean": float(scores.mean()),
        "sigma": float(es.sigma), "time_s": gen_time,
    })
    print(f"  Best so far: {best_score:.4f} | Gen max: {scores.max():.4f} | ({gen_time:.0f}s)")
    (HIGHLEVEL_DIR / "search_history.json").write_text(json.dumps(history, indent=2))

# ── Save final config ─────────────────────────────────────────────────────────
final_cfg = {
    "planner_type":      "mlp",
    "mlp_weights_path":  "planner_weights.npz",
    "mlp_hidden":        HIDDEN_SIZES,
    "stand_seconds":     1.0,
    "training_info": {
        "vx_max": _VX_MAX, "vx_min": _VX_MIN,
        "vy_lim": _VY_LIM, "yaw_lim": _YAW_LIM,
        "best_fitness": float(best_score),
        "generations": N_GENERATIONS, "population": POPULATION,
    },
}
(HIGHLEVEL_DIR / "planner_config.json").write_text(json.dumps(final_cfg, indent=2))
print(f"\nTraining complete. Best fitness: {best_score:.4f}")
print(f"Weights: {HIGHLEVEL_DIR / 'planner_weights.npz'}")

In [ ]:
# ── CELL 10: Save MLP to Drive ────────────────────────────────────────────────
HIGHLEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
drive_mlp     = DRIVE_BACKUP / "highlevel_mlp"

if drive_mlp.exists():
    shutil.rmtree(drive_mlp)
shutil.copytree(str(HIGHLEVEL_DIR), str(drive_mlp))
print("MLP saved to Drive:", drive_mlp)
print("Contents:", [f.name for f in drive_mlp.iterdir()])

In [ ]:
# ── CELL 10b: Restore MLP from Drive (if Colab restarted) ────────────────────
HIGHLEVEL_DIR = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
drive_mlp     = DRIVE_BACKUP / "highlevel_mlp"

if not HIGHLEVEL_DIR.exists() and drive_mlp.exists():
    shutil.copytree(str(drive_mlp), str(HIGHLEVEL_DIR))
    print("Restored MLP from Drive.")
elif HIGHLEVEL_DIR.exists():
    print("MLP already present locally.")
else:
    print("[error] No MLP found. Run Cell 9 first.")

PLANNER_CONFIG = HIGHLEVEL_DIR / "planner_config.json"
print("Planner config:", PLANNER_CONFIG)
print("Exists:", PLANNER_CONFIG.exists())

In [ ]:
# ── CELL 11: Full track evaluation (300 s, with video) ───────────────────────
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "low_level_train" / "best_checkpoint"
PLANNER_CONFIG = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp" / "planner_config.json"
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"

os.chdir(COURSE_REPO_DIR)
!python run_track_bonus.py \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --planner-config {PLANNER_CONFIG} \
    --config configs/colab_runtime_config.json \
    --output-dir {TRACK_EVAL_DIR} \
    --entry-name {TEAM_NAME} \
    --duration-seconds 300 \
    --render-every 10 \
    --render-fps 5

if (TRACK_EVAL_DIR / "results.json").exists():
    r = json.loads((TRACK_EVAL_DIR / "results.json").read_text())
    print("\n=== FINAL RESULTS ===")
    print(json.dumps(r["metrics"], indent=2))
    print("\n=== SCORES ===")
    print(json.dumps(r["scores"], indent=2))
else:
    print("[error] results.json not found")

In [ ]:
# ── CELL 12: Create submission.json ──────────────────────────────────────────
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "low_level_train" / "best_checkpoint"
PLANNER_CONFIG = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp" / "planner_config.json"

submission = {
    "team_name": TEAM_NAME,
    "track2_option": "leaderboard",
    "checkpoint_dir": "best_checkpoint",
    "planner_config": "planner_config.json",
    "planner_code": "track_bonus/planner.py",
    "planner_weights": "planner_weights.npz",
    "high_level_planner_type": "learned_mlp_cmaes",
    "track_eval": "track_eval/results.json",
    "notes": (
        "Low-level: Brax PPO trained from scratch (stage1 10M steps 0.8 m/s, "
        "stage2 5M steps up to 0.95 m/s). "
        "High-level: 5->32->16->3 MLP (VX_MAX=0.85 m/s) trained with diagonal CMA-ES, "
        "in-process evaluation to avoid per-candidate JIT overhead, custom fitness = "
        "distance/200 + speed_bonus if lap completed. "
        "Failed: fine-tuning from best_checkpoint (wrong format for --restore-checkpoint-dir), "
        "training fast policy from scratch (unstable), CMA-ES with 15s eval (JIT overhead)."
    ),
}
(COURSE_REPO_DIR / "submission.json").write_text(json.dumps(submission, indent=2))
print((COURSE_REPO_DIR / "submission.json").read_text())

In [ ]:
# ── CELL 13: Copy submission artifacts to planner config location ─────────────
# The evaluator expects planner_config.json + planner_weights.npz at repo root
# (or wherever the submission says they are).

HIGHLEVEL_DIR  = COURSE_REPO_DIR / "artifacts" / "highlevel_mlp"
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "low_level_train" / "best_checkpoint"

# Copy planner config + weights to repo root for easy access
shutil.copy(str(HIGHLEVEL_DIR / "planner_config.json"), str(COURSE_REPO_DIR / "planner_config.json"))
shutil.copy(str(HIGHLEVEL_DIR / "planner_weights.npz"), str(COURSE_REPO_DIR / "planner_weights.npz"))
print("Copied planner_config.json and planner_weights.npz to repo root.")

# Copy best_checkpoint to repo root
dest_ckpt = COURSE_REPO_DIR / "best_checkpoint"
if dest_ckpt.exists():
    shutil.rmtree(dest_ckpt)
shutil.copytree(str(CHECKPOINT_DIR), str(dest_ckpt))
print("Copied best_checkpoint/ to repo root.")

# Copy track_eval dir to repo root
TRACK_EVAL_DIR = COURSE_REPO_DIR / "artifacts" / "track_eval"
dest_eval = COURSE_REPO_DIR / "track_eval"
if dest_eval.exists():
    shutil.rmtree(dest_eval)
shutil.copytree(str(TRACK_EVAL_DIR), str(dest_eval))
print("Copied track_eval/ to repo root.")

In [ ]:
# ── CELL 14: Final checklist + upload to Drive ────────────────────────────────
expected = {
    "best_checkpoint/":          COURSE_REPO_DIR / "best_checkpoint",
    "planner_config.json":       COURSE_REPO_DIR / "planner_config.json",
    "planner_weights.npz":       COURSE_REPO_DIR / "planner_weights.npz",
    "track_bonus/planner.py":    COURSE_REPO_DIR / "track_bonus" / "planner.py",
    "track_eval/results.json":   COURSE_REPO_DIR / "track_eval" / "results.json",
    "submission.json":           COURSE_REPO_DIR / "submission.json",
}
all_ok = True
for label, path in expected.items():
    ok = path.exists()
    if not ok:
        all_ok = False
    print("OK" if ok else "MISSING", label)

if all_ok:
    print("\nAll artifacts present.")
    # Package everything to Drive
    drive_final = DRIVE_BACKUP / "final_submission"
    if drive_final.exists():
        shutil.rmtree(drive_final)
    drive_final.mkdir(parents=True)
    for label, path in expected.items():
        dest = drive_final / label.rstrip("/")
        dest.parent.mkdir(parents=True, exist_ok=True)
        if path.is_dir():
            shutil.copytree(str(path), str(dest))
        else:
            shutil.copy(str(path), str(dest))
    print(f"Packaged to Drive: {drive_final}")
else:
    print("\n[warn] Some artifacts missing. Complete the steps above first.")

# Print final scores
results_path = COURSE_REPO_DIR / "track_eval" / "results.json"
if results_path.exists():
    r = json.loads(results_path.read_text())
    print("\n=== FINAL COMPOSITE SCORE ===")
    print(f"  composite_score : {r['scores']['composite_score']:.4f}")
    print(f"  lap_completion  : {r['metrics']['lap_completion']}")
    print(f"  finish_time     : {r['metrics']['finish_time']}")
    print(f"  fall            : {r['metrics']['fall']}")

In [ ]:
# ── CELL 15: Push results back to GitHub ─────────────────────────────────────
# Run this LAST after all artifacts are ready.
# You'll need to authenticate git with your GitHub token.

# First: set up git credentials (only needed once per Colab session)
# !git config --global user.email "jrzzhang@ucdavis.edu"
# !git config --global user.name "Jiarao Zhang"

# Then push (replace YOUR_TOKEN with a GitHub personal access token):
# import getpass
# token = getpass.getpass("GitHub token: ")
# REPO_URL_WITH_TOKEN = f"https://{token}@github.com/jiarao76/Final-Project-Track-2-Bonus-Project.git"

os.chdir(COURSE_REPO_DIR)

# Stage submission artifacts
!git add submission.json track_bonus/planner.py planner_config.json planner_weights.npz
!git add best_checkpoint/ track_eval/ 2>/dev/null || true
!git status --short

# Commit (uncomment and run)
# !git commit -m "Add trained MLP planner and track evaluation results"

# Push (uncomment and fill in token)
# !git push {REPO_URL_WITH_TOKEN} main
print("Uncomment the commit/push lines above when ready.")